# CardioIA — Fase 4 — Parte 1: Pré-processamento de imagens

**Dataset:** CAMUS (ecocardiografia) — mesmo da Fase 1, organizado no Google Drive.

**Estrutura no Drive:**
```
dataset_cardio/
├── frames/    ← 900 imagens  (frame_0000.png …)
└── masks/     ← 900 máscaras (mask_0000.png …)
```

**Tarefa de classificação (enunciado FIAP):** classificar imagens médicas com CNN.

**Nossa tarefa:** prever, a partir **somente da imagem**, qual **estrutura cardíaca predominante** aparece na segmentação expert do CAMUS. Os rótulos são extraídos das **máscaras** (valores 253, 254, 255); na inferência o modelo **não** usa a máscara.

### Google Colab — ordem de execução

1. Dependências → montar Drive → configuração
2. Montar índice (frame + máscara + rótulo)
3. Exploração visual → pré-processamento → splits → salvar CSVs

> Protótipo acadêmico — simulação educacional; não substitui diagnóstico médico.

### Dependências

Execute uma vez por sessão no Colab.

In [ ]:
!pip install -q pandas numpy matplotlib seaborn pillow scikit-learn opencv-python-headless

### Montar Google Drive

Suas pastas `frames/` e `masks/` devem estar em **Meu Drive → `dataset_cardio`**.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

CAMINHO_BASE = Path("/content/drive/MyDrive/dataset_cardio")
CAMINHO_FRAMES = CAMINHO_BASE / "frames"
CAMINHO_MASKS = CAMINHO_BASE / "masks"

print("Base:", CAMINHO_BASE.exists(), CAMINHO_BASE)
print("Frames:", CAMINHO_FRAMES.exists(), len(list(CAMINHO_FRAMES.glob("*.png"))), "PNG")
print("Masks:", CAMINHO_MASKS.exists(), len(list(CAMINHO_MASKS.glob("*.png"))), "PNG")

### Configuração e imports

In [ ]:
from __future__ import annotations

import json

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from sklearn.model_selection import train_test_split

SEED = 42
IMG_SIZE = (224, 224)
PROP_TREINO, PROP_VAL, PROP_TESTE = 0.70, 0.15, 0.15

# Valores de pixel nas máscaras CAMUS (Kaggle) — 3 regiões anotadas
PIXELS_ESTRUTURA = (253, 254, 255)
NOMES_CLASSE = {253: "estrutura_253", 254: "estrutura_254", 255: "estrutura_255"}

# False = 3 classes (253/254/255) | True = binário (254 predominante vs demais)
MODO_BINARIO = False

np.random.seed(SEED)

SAIDA = CAMINHO_BASE
ARQUIVO_INDICE = SAIDA / "camus_indice_imagens.csv"
ARQUIVO_SPLITS = SAIDA / "camus_splits.csv"
ARQUIVO_CONFIG = SAIDA / "camus_preprocess_config.json"

print("Modo:", "binário (254 vs outras)" if MODO_BINARIO else "3 classes (253/254/255)")
print("Saída CSV:", SAIDA)

### Montar índice: frames + máscaras + rótulos

Para cada `frame_XXXX.png`, buscamos `mask_XXXX.png` e definimos o rótulo pela **região com mais pixels** na máscara (excluindo fundo 0).

In [ ]:
def caminho_mascara(frame: Path) -> Path:
    """frame_0007.png → mask_0007.png na pasta masks/."""
    sufixo = frame.stem.replace("frame_", "")
    return CAMINHO_MASKS / f"mask_{sufixo}.png"


def pixel_predominante(mascara: np.ndarray) -> int:
    contagem = {v: int((mascara == v).sum()) for v in PIXELS_ESTRUTURA}
    return max(contagem, key=contagem.get)


def classe_final(pixel: int) -> tuple[str, int]:
    if MODO_BINARIO:
        nome = "predominante_254" if pixel == 254 else "outras_estruturas"
        codigo = 1 if pixel == 254 else 0
        return nome, codigo
    return NOMES_CLASSE[pixel], pixel


def montar_indice() -> pd.DataFrame:
    registros = []
    frames = sorted(CAMINHO_FRAMES.glob("frame_*.png"))

    for frame in frames:
        mask = caminho_mascara(frame)
        if not mask.exists():
            print(f"⚠ Máscara ausente: {mask.name}")
            continue
        ms = np.array(Image.open(mask))
        px = pixel_predominante(ms)
        classe, rotulo = classe_final(px)
        registros.append(
            {
                "arquivo_frame": frame.name,
                "arquivo_mask": mask.name,
                "caminho_frame": str(frame.resolve()),
                "caminho_mask": str(mask.resolve()),
                "pixel_predominante": px,
                "classe": classe,
                "rotulo": rotulo,
                "fonte_rotulo": "mascara_camus",
            }
        )
    return pd.DataFrame(registros)


df = montar_indice()
print(f"Pares frame/máscara: {len(df)}")
if len(df) > 0:
    display(df.head())
    print("\nDistribuição por classe:")
    print(df["classe"].value_counts())
    print("\nPixels predominantes:")
    print(df["pixel_predominante"].value_counts())

### Exploração visual

Conferimos imagem + máscara + rótulo gerado. Isso documenta a origem dos labels no relatório.

In [ ]:
if len(df) == 0:
    print("Configure CAMINHO_FRAMES e CAMINHO_MASKS no Drive.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    sns.countplot(data=df, x="classe", ax=axes[0], palette="viridis")
    axes[0].set_title("Distribuição das classes")
    axes[0].tick_params(axis="x", rotation=20)
    sns.countplot(data=df, x="pixel_predominante", ax=axes[1], palette="magma")
    axes[1].set_title("Pixel predominante na máscara")
    plt.tight_layout()
    plt.show()

    amostras = df.groupby("classe", group_keys=False).apply(
        lambda g: g.sample(min(1, len(g)), random_state=SEED)
    )
    n = len(amostras)
    fig, axes = plt.subplots(n, 2, figsize=(8, 3 * n))
    if n == 1:
        axes = np.array([axes])
    for i, (_, row) in enumerate(amostras.iterrows()):
        frame = np.array(Image.open(row["caminho_frame"]).convert("L"))
        mask = np.array(Image.open(row["caminho_mask"]))
        axes[i, 0].imshow(frame, cmap="gray")
        axes[i, 0].set_title(f"Frame — {row['arquivo_frame']}")
        axes[i, 0].axis("off")
        axes[i, 1].imshow(mask, cmap="gray")
        axes[i, 1].set_title(f"Máscara → {row['classe']}")
        axes[i, 1].axis("off")
    plt.suptitle("Amostras por classe (rótulo da máscara)")
    plt.tight_layout()
    plt.show()

### Pré-processamento

- **Redimensionamento** 224×224 (compatível com VGG16/ResNet na Parte 2)
- **RGB** (3 canais para transfer learning)
- **Normalização** [0, 1]

In [ ]:
def carregar_imagem(caminho: str | Path) -> np.ndarray:
    with Image.open(caminho) as img:
        return np.array(img.convert("RGB"))


def redimensionar(imagem: np.ndarray, tamanho: tuple[int, int] = IMG_SIZE) -> np.ndarray:
    return cv2.resize(imagem, tamanho, interpolation=cv2.INTER_AREA)


def normalizar(imagem: np.ndarray) -> np.ndarray:
    return (imagem.astype(np.float32) / 255.0)


def preprocessar(caminho: str | Path) -> np.ndarray:
    img = carregar_imagem(caminho)
    img = redimensionar(img)
    return normalizar(img)


if len(df) > 0:
    exemplo = df.iloc[0]
    original = carregar_imagem(exemplo["caminho_frame"])
    processada = preprocessar(exemplo["caminho_frame"])
    print(
        f"Original {original.shape} → processada {processada.shape} | "
        f"min/max {processada.min():.2f}/{processada.max():.2f} | classe {exemplo['classe']}"
    )
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(original)
    axes[0].set_title("Original (RGB)")
    axes[0].axis("off")
    axes[1].imshow(processada)
    axes[1].set_title(f"Resize {IMG_SIZE} + norm [0,1]")
    axes[1].axis("off")
    plt.show()

### Splits treino / validação / teste

Divisão **estratificada** por `classe`, seed fixa, proporção **70% / 15% / 15%**.

In [ ]:
if len(df) == 0:
    print("Sem dados — verifique as pastas no Drive.")
else:
    treino_val, teste = train_test_split(
        df,
        test_size=PROP_TESTE,
        random_state=SEED,
        stratify=df["rotulo"],
    )
    proporcao_val = PROP_VAL / (PROP_TREINO + PROP_VAL)
    treino, val = train_test_split(
        treino_val,
        test_size=proporcao_val,
        random_state=SEED,
        stratify=treino_val["rotulo"],
    )

    treino = treino.assign(split="treino")
    val = val.assign(split="validacao")
    teste = teste.assign(split="teste")
    df_splits = pd.concat([treino, val, teste], ignore_index=True)

    print("Tamanhos:", df_splits["split"].value_counts().to_dict())
    print("\nPor classe em cada split:")
    print(pd.crosstab(df_splits["split"], df_splits["classe"]))

    assert df_splits.groupby("caminho_frame")["split"].nunique().max() == 1
    print("\n✓ Sem vazamento entre splits.")

### Salvar índices e configuração

CSVs ficam em `dataset_cardio/` no Drive para a **Parte 2** reutilizar os mesmos splits.

In [ ]:
if len(df) > 0:
    df.to_csv(ARQUIVO_INDICE, index=False, encoding="utf-8")
    df_splits.to_csv(ARQUIVO_SPLITS, index=False, encoding="utf-8")

    classes = sorted(df["classe"].unique())
    config = {
        "dataset": "CAMUS",
        "fonte": "Kaggle parsakh / Fase 1 CardioIA",
        "tarefa": "classificacao_estrutura_predominante",
        "modo_binario": MODO_BINARIO,
        "classes": classes,
        "pixels_mascara": list(PIXELS_ESTRUTURA),
        "img_size": list(IMG_SIZE),
        "normalizacao": "0_a_1",
        "canais": "RGB",
        "seed": SEED,
        "split": {"treino": PROP_TREINO, "validacao": PROP_VAL, "teste": PROP_TESTE},
        "total_imagens": int(len(df)),
        "caminho_frames": str(CAMINHO_FRAMES.resolve()),
        "caminho_masks": str(CAMINHO_MASKS.resolve()),
        "nota": "Rotulos derivados das mascaras; inferencia usa apenas frames.",
    }
    ARQUIVO_CONFIG.write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")

    print("Arquivos salvos:")
    print(" -", ARQUIVO_INDICE)
    print(" -", ARQUIVO_SPLITS)
    print(" -", ARQUIVO_CONFIG)

### Conclusão (Parte 1)

**Pipeline:** Drive `frames/` + `masks/` → rótulos da máscara → resize 224×224 → RGB → [0,1] → splits estratificados → CSVs.

**Parte 2:** treinar CNN simples + transfer learning com `camus_splits.csv` (entrada = `caminho_frame`; **sem** máscara na predição).

**Relatório:** documentar origem dos rótulos, desbalanceamento das classes e limitações do protótipo acadêmico.